# Actividad 04 — Arquitectura Medallón con Datos de Fraude

**Semana:** 02  
**Capa:** Bronze  
**Notebook:** bronze_daniel  
**Objetivo:** Ingestar las 5 fuentes del dataset financiero como tablas Delta sin transformar los datos.

## Diseño del pipeline

### Fuentes → Bronze

transactions_data.csv   → bronze.transactions_daniel  
users_data.csv          → bronze.users_daniel  
cards_data.csv          → bronze.cards_daniel  
mcc_codes.json          → bronze.mcc_codes_daniel  
train_fraud_labels.json → bronze.fraud_labels_daniel  

### Bronze → Silver

bronze.transactions_daniel  
bronze.users_daniel  
bronze.cards_daniel  
bronze.mcc_codes_daniel  
bronze.fraud_labels_daniel  
↓  
limpieza de tipos  
estandarización de columnas  
pivot de mcc_codes  
JOINs entre las 5 tablas  
validación de duplicados  
↓  
silver.transactions_daniel  

### Silver → Gold

silver.transactions_daniel  
↓  
gold.fraude_por_categoria_daniel  
gold.fraude_por_tarjeta_daniel  
gold.fraude_temporal_daniel  
gold.usuarios_riesgo_daniel  

## Reglas por capa

**Bronze:** ingesta fiel de la fuente, sin transformar datos.  
**Silver:** limpieza, estandarización, JOINs y validaciones.  
**Gold:** tablas agregadas orientadas a preguntas de negocio.

In [0]:
MI_NOMBRE = "daniel"
VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"

display(dbutils.fs.ls(VOL))

In [0]:
from pyspark.sql import functions as F

MI_NOMBRE = "daniel"
CATALOG = "workspace"
VOL = f"/Volumes/workspace/default/week_2_{MI_NOMBRE}"

# Usar catálogo workspace
spark.sql(f"USE CATALOG {CATALOG}")

# Crear schemas si no existen
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

# 1. transactions — CSV sin inferir schema para conservar Bronze fiel
df_bronze_tx = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{VOL}/transactions_data.csv")
)

df_bronze_tx.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.bronze.transactions_{MI_NOMBRE}"
)

print(f"{CATALOG}.bronze.transactions_{MI_NOMBRE}: {df_bronze_tx.count():,} filas | {len(df_bronze_tx.columns)} columnas")


# 2. users
df_bronze_users = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{VOL}/users_data.csv")
)

df_bronze_users.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.bronze.users_{MI_NOMBRE}"
)

print(f"{CATALOG}.bronze.users_{MI_NOMBRE}: {df_bronze_users.count():,} filas | {len(df_bronze_users.columns)} columnas")


# 3. cards
df_bronze_cards = (
    spark.read.format("csv")
    .option("header", "true")
    .option("inferSchema", "false")
    .load(f"{VOL}/cards_data.csv")
)

df_bronze_cards.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.bronze.cards_{MI_NOMBRE}"
)

print(f"{CATALOG}.bronze.cards_{MI_NOMBRE}: {df_bronze_cards.count():,} filas | {len(df_bronze_cards.columns)} columnas")


# 4. mcc_codes — JSON
df_bronze_mcc = (
    spark.read
    .option("multiLine", "true")
    .json(f"{VOL}/mcc_codes.json")
)

df_bronze_mcc.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.bronze.mcc_codes_{MI_NOMBRE}"
)

print(f"{CATALOG}.bronze.mcc_codes_{MI_NOMBRE}: {df_bronze_mcc.count():,} filas | {len(df_bronze_mcc.columns)} columnas")


# 5. fraud_labels — Parquet por performance
# Nota: el JSON existe, pero en Databricks Free/Serverless puede tardar demasiado.
# El dataset también incluye train_fraud_labels.parquet, que conserva la misma información
# y permite crear la tabla Bronze de forma más eficiente.
df_bronze_fraud = spark.read.parquet(f"{VOL}/train_fraud_labels.parquet")

df_bronze_fraud.write.format("delta").mode("overwrite").saveAsTable(
    f"{CATALOG}.bronze.fraud_labels_{MI_NOMBRE}"
)

print(f"{CATALOG}.bronze.fraud_labels_{MI_NOMBRE}: {df_bronze_fraud.count():,} filas | {len(df_bronze_fraud.columns)} columnas")


print("\nBronze completo. Tablas disponibles:")
display(spark.sql(f"SHOW TABLES IN {CATALOG}.bronze"))

## Observación de performance en Bronze

Inicialmente se intentó leer `train_fraud_labels.json`, pero el archivo tardó demasiado en procesarse en Databricks Free/Serverless.

Como el dataset también incluye `train_fraud_labels.parquet`, se usó esta versión para crear `workspace.bronze.fraud_labels_daniel`.

Parquet conserva la información necesaria para continuar la pipeline y permite procesar la capa Bronze de forma más eficiente en este entorno.